# Run New Signals: Polarisation & Centroid Distance

This notebook processes all 100 matches through the two new signals and saves
their outputs. It then runs `merge_outputs.py` to update the unified dataset.

**Signals processed:**
1. `team_polarisation` — Movement vector alignment per team (mean resultant vector length R)
2. `team_centroid_distance` — Per-player distance from team centre of mass

**Note:** This notebook is designed to run on Conor's laptop with full data.
For Pi testing, use `nrows=500` to avoid OOM.

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────
# Add the project root to sys.path and import dependencies

import sys
from pathlib import Path

# Adjust this if your notebook is elsewhere
PROJECT_ROOT = Path.cwd().parent  # Assumes notebook is in notebooks/
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

import logging
import pandas as pd
import numpy as np

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)
logger = logging.getLogger(__name__)
print("Setup complete.")

In [ ]:
# ── Cell 2: Import signals (triggers registration) ────────────────────
# Importing the signal modules activates the @register_signal decorator

import src.signals.polarisation    # noqa: F401
import src.signals.team_centroid_distance  # noqa: F401

from src.signals.registry import discover_signals, get_signal_names

# Verify registration
names = get_signal_names()
print(f"Registered signals: {names}")

In [ ]:
# ── Cell 3: Define batch processing function ─────────────────────────
# This function processes all matches for the new signals

from src.model1.base import get_available_match_ids, load_match_tracking
from src.model1.block_builder import build_blocks
from src.smoothing import compute_velocity_features
from src.config import SIGNAL_OUTPUT_DIR


def process_new_signals(
    match_ids: list[int],
    nrows: int = None,
    output_dir: str = None,
) -> dict:
    """Process all matches for the two new signals.

    Parameters
    ----------
    match_ids : list[int]
        Match IDs to process.
    nrows : int, optional
        Limit tracking rows per match (for testing).
    output_dir : str, optional
        Output directory for signal CSVs. Defaults to SIGNAL_OUTPUT_DIR.

    Returns
    -------
    dict
        signal_name -> combined DataFrame for all matches.
    """
    from pathlib import Path

    out = Path(output_dir or SIGNAL_OUTPUT_DIR)
    out.mkdir(parents=True, exist_ok=True)

    signals = discover_signals()
    combined = {name: [] for name in signals}

    for i, match_id in enumerate(match_ids):
        print(f"[{i+1}/{len(match_ids)}] Processing match {match_id}...")

        # Load tracking data
        df = load_match_tracking(match_id, nrows=nrows)
        if df is None:
            print(f"  ⚠️  No tracking data for match {match_id}, skipping")
            continue

        # Compute velocity features
        if all(c in df.columns for c in ["x", "y", "frame", "player_id"]):
            df = compute_velocity_features(df)

        # Build blocks
        blocks = build_blocks(df, match_id)
        if not blocks:
            print(f"  ⚠️  No blocks for match {match_id}, skipping")
            continue

        # Run each signal
        for name, cls in signals.items():
            try:
                sig = cls()
                result = sig.compute(blocks, match_id=match_id)
                if result is not None and not result.empty:
                    # Save individual match output
                    match_path = out / f"signal_{name}_match_{match_id}.csv"
                    result.to_csv(match_path, index=False)
                    combined[name].append(result)
                    print(f"  ✅ {name}: {len(result)} rows")
                else:
                    print(f"  ⚠️  {name}: empty result")
            except Exception as e:
                print(f"  ❌ {name}: {e}")

    # Combine all matches for each signal
    combined_dfs = {}
    for name, dfs in combined.items():
        if dfs:
            combined_dfs[name] = pd.concat(dfs, ignore_index=True)
            combined_path = out / f"signal_{name}_all_matches.csv"
            combined_dfs[name].to_csv(combined_path, index=False)
            print(f"Combined {name}: {len(combined_dfs[name])} rows → {combined_path}")

    return combined_dfs

In [ ]:
# ── Cell 4: Discover match IDs ───────────────────────────────────────
# Find all available matches (up to 100)

match_ids = get_available_match_ids(limit=100)
print(f"Found {len(match_ids)} matches: {match_ids[:5]}...{match_ids[-3:]}")

# ── QUICK TEST (Pi-safe): use nrows=500 ─────────────────────────────
# On the Raspberry Pi, set nrows=500 to avoid OOM.
# On your laptop, set nrows=None to load full data.
NROWS = None   # <-- CHANGE TO None ON LAPTOP, 500 ON PI
print(f"NROWS = {NROWS}  (None = full data, 500 = Pi-safe test)")

In [ ]:
# ── Cell 5: Run the pipeline ─────────────────────────────────────────
# This processes all matches and saves individual CSVs
# Estimated time: ~15–30 minutes for 100 matches with full data

from src.config import SIGNAL_OUTPUT_DIR

print("Starting signal processing...")
print(f"Matches: {len(match_ids)}")
print(f"Output: {SIGNAL_OUTPUT_DIR}")
print()

combined = process_new_signals(match_ids, nrows=NROWS)

print()
print("Signal processing complete!")
for name, df in combined.items():
    print(f"  {name}: {len(df)} total rows")

In [ ]:
# ── Cell 6: Inspect signal outputs ───────────────────────────────────
# Quick sanity check on the output distributions

for name, df in combined.items():
    print(f"\n{'='*50}")
    print(f"Signal: {name}")
    print(f"{'='*50}")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print()
    print("Signal value distribution:")
    print(df["signal_value"].describe())
    print()
    print("Unique teams:", df["team_id_opta"].nunique())
    print("Unique blocks:", df["block_id"].nunique())
    print("Unique phases:", df["phase"].unique())

In [ ]:
# ── Cell 7: Run merge_outputs.py to update unified dataset ───────────
# This merges the new signal outputs with existing Model 1 and Model 2
# outputs into a single unified_dataset.csv

%run -m src.merge_outputs

# Verify the output
from src.config import OUTPUT_DIR
unified_path = OUTPUT_DIR / "unified_dataset.csv"
if unified_path.exists():
    unified = pd.read_csv(unified_path)
    print(f"\nUnified dataset: {len(unified)} rows x {len(unified.columns)} columns")
    print(f"Columns: {list(unified.columns)}")
else:
    print(f"⚠️  Unified dataset not found at {unified_path}")

In [ ]:
# ── Cell 8: Run opponent quality covariate analysis (optional) ────────
# Uncomment the lines below to run the post-hoc analysis

# %run ../analysis/opponent_quality_covariate.py

# print("Opponent quality analysis complete!")
# results = pd.read_csv("../analysis/opponent_quality_results.csv")
# display(results)

In [ ]:
# ── Cell 9: Summary ───────────────────────────────────────────────────
# Final summary of what was processed

print("=" * 60)
print("NEW SIGNALS PIPELINE — COMPLETE")
print("=" * 60)
print()
print(f"Matches processed: {len(match_ids)}")
print(f"Signals run: {list(combined.keys())}")
print(f"Output directory: {SIGNAL_OUTPUT_DIR}")
print()
for name, df in combined.items():
    print(f"  {name}: {len(df)} rows across {df['game_id'].nunique()} matches")
print()
print("Next steps:")
print("  1. Run analysis/opponent_quality_covariate.py for post-hoc analysis")
print("  2. Feed unified dataset into dissociation analysis")
print("  3. Review signal distributions for outliers")